In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from learntools.core import binder
binder.bind(globals())
from learntools.sql.ex6 import *
print("Setup Complete")

In [ ]:
from google.cloud import bigquery

In [ ]:
client = bigquery.Client()

# Gettting Started with SQL&BigQuery

In [ ]:
# Construct a reference to the "hacker news" dataset
dataset_ref = client.dataset("hacker_news", project="bigquery-public-data")
# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

In [ ]:
# List all the tables in the "hacker_news" dataset
tables = list(client.list_tables(dataset))
# Print names of all tables in the dataset
for table in tables:
    print(table.table_id)

In [ ]:
# Construct a reference to the "full" table
table_ref = dataset_ref.table("full")
# API request - fetch the table
table = client.get_table(table_ref)

In [ ]:
# Print information on all the columns in the "full" table in the "hacker_news" dataset
table.schema

In [ ]:
# preview the first five lines of the full table
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
# Preview the first five entries in the "by" column of the "full" table
client.list_rows(table, selected_fields=table.schema[:1], max_results=5).to_dataframe()

## Exercise 1

In [ ]:
dataset_ref = client.dataset("chicago_crime", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)
tables = list(client.list_tables(dataset))
num_tables = len(tables)
print(num_tables)
print(table.table_id)

In [ ]:
q_3.check()

In [ ]:
# Construct a reference to the "full" table
table_ref = dataset_ref.table("crime")
# API request - fetch the table
table = client.get_table(table_ref)

table.schema
num_timestamp_fields = 2

In [ ]:
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
client.list_rows(table, selected_fields=table.schema[19:], max_results=5).to_dataframe()

In [ ]:
fields_for_plotting = ['latitude','longitude']

# Select, From & Where

In [ ]:
dataset_ref = client.dataset("openaq", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

In [ ]:
table_ref = dataset_ref.table("global_air_quality")
table = client.get_table(table_ref)
table.schema

In [ ]:
client.list_rows(table, max_results=10).to_dataframe()

In [ ]:
query = """"""
        SELECT city, country
        FROM `bigquery-public-data.openaq.global_air_quality`
        WHERE country = "US"
        """"""

In [ ]:
# Set up the query
query_job = client.query(query)

In [ ]:
# API request - run the query, and return a pandas DataFrame
us_cities = query_job.to_dataframe()
us_cities

In [ ]:
# What five cities have the most measurements?
us_cities.city.value_counts().head()

## Working with big datasets

In [ ]:
# Query to get the score column from every row where the type column has value "job"
query = """"""
        SELECT score
        FROM `bigquery-public-data.hacker_news.full`
        WHERE type = "job"
        """"""

# Create a QueryJobConfig object to estimate size of query without running it
dry_run_config = bigquery.QueryJobConfig(dry_run=True)
# API request - dry run query to estimate costs
dry_run_query_job = client.query(query, job_config=dry_run_config)

print("This query will process {} bytes.".format(dry_run_query_job.total_bytes_processed))

## Exercise 2

In [ ]:
dataset_ref = client.dataset("openaq", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("global_air_quality")
table = client.get_table(table_ref)
table.schema

client.list_rows(table, max_results=2).to_dataframe()

In [ ]:
# Which countries have reported pollution levels in units of "ppm"?
first_query = """
              SELECT country
              FROM `bigquery-public-data.openaq.global_air_quality`
              WHERE unit = "ppm"
              """

first_query_job = client.query(first_query)
first_result = first_query_job.to_dataframe()
first_result

In [ ]:
q_2.check()

In [ ]:
zero_pollution_query = """
                       SELECT *
                       FROM `bigquery-public-data.openaq.global_air_quality`
                       WHERE value = 0
                       """
query_job = client.query(zero_pollution_query)
zero_pollution_results = query_job.to_dataframe()
zero_pollution_results.head()

# Group By, Having & Count

In [ ]:
dataset_ref = client.dataset("hacker_news", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("full")
table = client.get_table(table_ref)
table.schema
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
query = """
        SELECT parent, COUNT(id)
        FROM `bigquery-public-data.hacker_news.full`
        GROUP BY parent
        HAVING COUNT(id) > 10
        """
query_job = client.query(query)
popular_comments = query_job.to_dataframe()
popular_comments.head()

In [ ]:
# Improved
query = """
        SELECT parent, COUNT(1) AS NumPosts
        FROM `bigquery-public-data.hacker_news.full`
        GROUP BY parent
        HAVING COUNT(id) > 10
        """
query_job = client.query(query)
popular_comments = query_job.to_dataframe()
popular_comments.head()

## Exercise

In [ ]:
# Write a query that returns all authors with more than 10,000 posts as well as their post counts
prolific_commenters_query = """
                            SELECT `by` author, COUNT(id) AS NumPosts
                            FROM `bigquery-public-data.hacker_news.full`
                            GROUP BY author
                            HAVING COUNT(id) > 10000
                            """

query_job = client.query(prolific_commenters_query)
prolific_commenters = query_job.to_dataframe()
prolific_commenters

In [ ]:
# How many comments have been deleted? (If a comment was deleted, the deleted column in the table will have the value True.)
num_deleted_query = """
                    SELECT COUNT(1) as nums_deleted_posts
                    FROM `bigquery-public-data.hacker_news.full`
                    WHERE deleted = True
                    """

query_job = client.query(num_deleted_query)
num_deleted_posts = query_job.to_dataframe()
print(num_deleted_posts)

# ORDER BY

In [ ]:
dataset_ref = client.dataset("nhtsa_traffic_fatalities", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("accident_2015")
table = client.get_table(table_ref)

client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
query = """
        SELECT EXTRACT(DAYOFWEEK from timestamp_of_crash) AS day_of_week,
               COUNT(consecutive_number) AS num_accidents
        FROM `bigquery-public-data.nhtsa_traffic_fatalities.accident_2015`
        GROUP BY day_of_week
        ORDER BY num_accidents DESC
        """

query_job = client.query(query)
accidents_by_day = query_job.to_dataframe()
accidents_by_day.head()

## Exercise

In [ ]:
dataset_ref = client.dataset("world_bank_intl_education", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("international_education")
table = client.get_table(table_ref)

client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
country_spend_pct_query = """
                          SELECT country_name, AVG(value) AS avg_ed_spending_pct 
                          FROM `bigquery-public-data.world_bank_intl_education.international_education`
                          WHERE indicator_code = 'SE.XPD.TOTL.GD.ZS'and year >= 2010 and year <= 2017
                          GROUP BY country_name
                          ORDER BY avg_ed_spending_pct DESC
                          """

query_job = client.query(country_spend_pct_query)
country_spending_results  = query_job.to_dataframe()
country_spending_results.head()

In [ ]:
code_count_query = """
                          SELECT indicator_code, indicator_name, COUNT(1) AS num_rows
                          FROM `bigquery-public-data.world_bank_intl_education.international_education`
                          WHERE year = 2016
                          GROUP BY indicator_code, indicator_name
                          HAVING num_rows >= 175
                          ORDER BY num_rows DESC
                          """

query_job = client.query(code_count_query)
code_count_results  = query_job.to_dataframe()
code_count_results.head()

## AS&WITH

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

dataset_ref = client.dataset("crypto_bitcoin", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)
tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("transactions")
table = client.get_table(table_ref)

client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
# Query to select the number of transactions per date, sorted by date
query_with_CTE = """
                 With time AS
                 (
                 SELECT DATE(block_timestamp) AS trans_date
                 FROM `bigquery-public-data.crypto_bitcoin.transactions`
                 )
                 SELECT COUNT(1) AS transactions, trans_date
                 FROM time
                 GROUP BY trans_date
                 ORDER BY trans_date
                 """
query_job = client.query(query_with_CTE)
transactions_by_date = query_job.to_dataframe()
transactions_by_date.head()

In [ ]:
transactions_by_date.set_index('trans_date').plot()

## Exercise

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

dataset_ref = client.dataset("chicago_taxi_trips", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("taxi_trips")
table = client.get_table(table_ref)
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
# Determine when this data is from
rides_per_year_query = """
                       SELECT EXTRACT(YEAR FROM trip_start_timestamp) AS year,
                              COUNT(1) AS num_trips,
                       FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                       GROUP BY year
                       ORDER BY year
                       """
query_job = client.query(rides_per_year_query)
rides_per_year_result = query_job.to_dataframe()
rides_per_year_result

In [ ]:
# Dive slightly deeper
rides_per_month_query = """
                       SELECT EXTRACT(MONTH FROM trip_start_timestamp) AS month,
                              COUNT(1) AS num_trips,
                       FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                       WHERE EXTRACT( YEAR FROM trip_start_timestamp) = 2016
                       GROUP BY month
                       ORDER BY month
                       """
query_job = client.query(rides_per_month_query)
rides_per_month_result = query_job.to_dataframe()
rides_per_month_result

In [ ]:
# Write the query
speeds_query = """
               WITH RelevantRides AS
               (
                   SELECT EXTRACT(HOUR FROM trip_start_timestamp) AS hour_of_day,
                          trip_miles,
                          trip_seconds
                   FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
                   WHERE trip_start_timestamp > '2016-01-01' and trip_start_timestamp < '2016-04-01' 
                   and trip_seconds > 0 and trip_miles > 0
               )
               SELECT COUNT(1) AS num_trips,
               hour_of_day,
               3600 * SUM(trip_miles) / SUM(trip_seconds) AS avg_mph
               FROM RelevantRides
               GROUP BY hour_of_day
               ORDER BY hour_of_day
               """

query_job = client.query(speeds_query)
speeds_result = query_job.to_dataframe()
speeds_result

# Joining Data

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

dataset_ref = client.dataset("github_repos", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
for table in tables:
    print(table.table_id)

table_ref = dataset_ref.table("licenses")
table = client.get_table(table_ref)
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
table_ref = dataset_ref.table("sample_files")
table = client.get_table(table_ref)
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
# Query to determine the number of files per license, sorted by number of files
query = """
        SELECT L.license, COUNT(1) AS num_of_files
        FROM `bigquery-public-data.github_repos.sample_files` AS sf
        INNER JOIN `bigquery-public-data.github_repos.licenses` AS L
        ON sf.repo_name = L.repo_name
        GROUP BY L.license
        ORDER BY num_of_files DESC
        """

query_job = client.query(query)
file_count_by_license = query_job.to_dataframe()
file_count_by_license

## Exercise

In [ ]:
from google.cloud import bigquery
client = bigquery.Client()

dataset_ref = client.dataset("stackoverflow", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = client.list_tables(dataset)
list_of_tables = [table.table_id for table in tables]
list_of_tables

In [ ]:
table_ref = dataset_ref.table("posts_answers")
table = client.get_table(table_ref)
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
table_ref = dataset_ref.table("posts_questions")
table = client.get_table(table_ref)
client.list_rows(table, max_results=5).to_dataframe()

In [ ]:
# Selecting the right questions
questions_query = """
                  SELECT id, title, owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_questions`
                  WHERE tags LIKE '%bigquery%'
                  """

query_job = client.query(questions_query)
questions_results = query_job.to_dataframe()
questions_results.head()

In [ ]:
# Your first join
answers_query = """
                  SELECT a.id, a.body, a.owner_user_id
                  FROM `bigquery-public-data.stackoverflow.posts_answers` AS a
                  INNER JOIN `bigquery-public-data.stackoverflow.posts_questions` AS q
                  ON a.parent_id = q.id
                  WHERE q.tags LIKE '%bigquery%'
                  """
query_job = client.query(answers_query)
answers_results = query_job.to_dataframe()
answers_results.head()